# 🗣️ Lab Module 26 — Multi-Agent Debate & Consensus

**Mục tiêu:** Implement toàn bộ multi-agent debate system cho FinTech Corp LoanBot:
- Section 1: Debate Router — khi nào cần debate
- Section 2: Agent Personas — OptimistAgent, PessimistAgent, RegulatorAgent
- Section 3: Consensus Mechanisms — 4 methods (Majority, Weighted, Borda, Delphi)
- Section 4: Devil's Advocate Pattern
- Section 5: Meta-Arbitrator với Audit Trail
- Section 6: Panel Quality Metrics
- Section 7: Tổng hợp — LoanBotDebateSystem (TODO → Solution)

**Dữ liệu test:** 4 mock customers của FinTech Corp
- FC-2024-001: Score 720 (clear approve)
- FC-2024-002: Score 580 (borderline)
- FC-2024-003: Score 400 (clear reject)
- FC-2024-004: Score 645, DTI 0.42 (borderline + short employment)

In [ ]:
# Setup — không cần Redis, tất cả in-memory
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from enum import Enum
import random
import math

# Mock customers
APPLICATIONS = [
    {'id': 'FC-2024-001', 'credit_score': 720, 'dti_ratio': 0.30, 'loan_amount': 500_000_000, 'employment_months': 36, 'loan_purpose': 'Home purchase'},
    {'id': 'FC-2024-002', 'credit_score': 580, 'dti_ratio': 0.43, 'loan_amount': 300_000_000, 'employment_months': 14, 'loan_purpose': 'Personal consumption'},
    {'id': 'FC-2024-003', 'credit_score': 400, 'dti_ratio': 0.65, 'loan_amount': 200_000_000, 'employment_months': 6,  'loan_purpose': 'Unknown'},
    {'id': 'FC-2024-004', 'credit_score': 645, 'dti_ratio': 0.42, 'loan_amount': 400_000_000, 'employment_months': 8,  'loan_purpose': 'Business vehicle'},
]

class Decision(Enum):
    APPROVE = 'APPROVE'
    REVIEW  = 'REVIEW'
    REJECT  = 'REJECT'

print('Setup complete. 4 mock applications loaded.')

## Section 1: Debate Router
Implement hàm `route_to_debate()` — quyết định hồ sơ nào cần debate panel.

In [ ]:
@dataclass
class DebateRoute:
    use_debate: bool
    n_agents: int
    reason: str
    direct_decision: Optional[Decision] = None  # Nếu không cần debate

def route_to_debate(app: dict) -> DebateRoute:
    score = app['credit_score']
    dti   = app['dti_ratio']
    emp   = app['employment_months']
    amt   = app['loan_amount']

    # Clear reject — no debate needed
    if score < 500 or dti > 0.55:
        return DebateRoute(False, 0, 'Clear reject criteria', Decision.REJECT)

    # Clear approve — no debate needed
    if score >= 720 and dti <= 0.35 and emp >= 24:
        return DebateRoute(False, 0, 'Clear approve criteria', Decision.APPROVE)

    # High-value borderline → 5-agent
    if amt > 2_000_000_000 and 580 <= score <= 700:
        return DebateRoute(True, 5, 'High-value borderline')

    # Standard borderline → 3-agent
    if 580 <= score <= 680 or 0.38 <= dti <= 0.45:
        return DebateRoute(True, 3, 'Standard borderline zone')

    if emp < 12:
        return DebateRoute(True, 3, 'Short employment — regulatory check needed')

    return DebateRoute(False, 0, 'Sufficient signal', Decision.APPROVE)

# Test routing
print('=== Debate Routing Results ===')
for app in APPLICATIONS:
    r = route_to_debate(app)
    if r.use_debate:
        print(f"{app['id']} (score={app['credit_score']}, dti={app['dti_ratio']}): DEBATE ({r.n_agents} agents) — {r.reason}")
    else:
        print(f"{app['id']} (score={app['credit_score']}, dti={app['dti_ratio']}): DIRECT {r.direct_decision.value} — {r.reason}")

## Section 2: Agent Personas
3 specialized agents với different risk tolerances. Mỗi agent có `assess()` method trả về vote + reasoning.

In [ ]:
@dataclass
class AgentVote:
    agent_name: str
    vote: Decision
    confidence: float
    reasoning: str
    weight: float = 1.0
    compliance_flag: bool = False

class OptimistAgent:
    """Risk-tolerant: focuses on growth potential. Minimize false negatives."""
    name = 'OptimistAgent'
    weight = 1.0

    def assess(self, app: dict) -> AgentVote:
        score = app['credit_score']
        dti   = app['dti_ratio']
        purpose = app.get('loan_purpose', '')
        productive = any(p in purpose.lower() for p in ['business', 'home', 'education', 'vehicle'])

        if score >= 600 and dti <= 0.44 and productive:
            return AgentVote(self.name, Decision.APPROVE, 0.78,
                f'Score {score} acceptable. Productive purpose ({purpose}). DTI {dti} within tolerance.', self.weight)
        elif score >= 580:
            return AgentVote(self.name, Decision.REVIEW, 0.65,
                f'Score borderline ({score}). Potential good borrower — needs additional docs.', self.weight)
        else:
            return AgentVote(self.name, Decision.REJECT, 0.82,
                f'Score {score} below minimum threshold even for optimistic assessment.', self.weight)

class PessimistAgent:
    """Risk-averse: focuses on default prevention. Minimize false positives."""
    name = 'PessimistAgent'
    weight = 1.2

    def assess(self, app: dict) -> AgentVote:
        score = app['credit_score']
        dti   = app['dti_ratio']
        emp   = app['employment_months']

        risk_score = 0
        reasons = []
        if score < 650:   risk_score += 2; reasons.append(f'Score {score} below safe threshold 650')
        if dti > 0.40:    risk_score += 2; reasons.append(f'DTI {dti} high — limited buffer')
        if emp < 12:      risk_score += 1; reasons.append(f'Employment only {emp}mo — unstable')
        if dti > 0.43:    risk_score += 1; reasons.append(f'DTI {dti} approaching ceiling 0.45')

        if risk_score >= 4:
            return AgentVote(self.name, Decision.REJECT, 0.85,
                'Multiple risk factors: ' + '; '.join(reasons), self.weight)
        elif risk_score >= 2:
            return AgentVote(self.name, Decision.REVIEW, 0.75,
                'Moderate risk: ' + '; '.join(reasons), self.weight)
        else:
            return AgentVote(self.name, Decision.APPROVE, 0.70,
                f'Risk profile acceptable. Score {score}, DTI {dti}, Emp {emp}mo.', self.weight)

class RegulatorAgent:
    """Compliance-focused: TT39/2016 enforcement. Has compliance veto power."""
    name = 'RegulatorAgent'
    weight = 1.5

    KNOWN_PURPOSES = {'home purchase', 'business vehicle', 'education', 'working capital'}

    def assess(self, app: dict) -> AgentVote:
        score = app['credit_score']
        dti   = app['dti_ratio']
        emp   = app['employment_months']
        purpose = app.get('loan_purpose', '').lower()

        # TT39/2016 Điều 9: loan purpose must be documented
        if purpose in {'unknown', '', 'other'}:
            return AgentVote(self.name, Decision.REVIEW, 0.99,
                'COMPLIANCE FLAG: TT39/2016 Điều 9 — loan purpose not documented. Must clarify before approval.',
                self.weight, compliance_flag=True)

        # TT39/2016: DTI ceiling 0.45
        if dti > 0.45:
            return AgentVote(self.name, Decision.REJECT, 0.99,
                f'COMPLIANCE FLAG: DTI {dti} exceeds TT39/2016 maximum 0.45. Mandatory reject.',
                self.weight, compliance_flag=True)

        # Internal policy: short employment requires collateral
        conditions = []
        if emp < 12:
            conditions.append('Employment < 12mo: require collateral per internal policy')

        if score < 580:
            return AgentVote(self.name, Decision.REJECT, 0.90,
                f'Score {score} below regulatory minimum 580.', self.weight)

        if conditions:
            return AgentVote(self.name, Decision.REVIEW, 0.85,
                f'Compliant but conditions required: {"; ".join(conditions)}', self.weight)

        return AgentVote(self.name, Decision.APPROVE, 0.80,
            f'Compliant: score {score}, DTI {dti}, purpose documented, employment {emp}mo.', self.weight)

# Test agents on FC-2024-004
app = APPLICATIONS[3]  # FC-2024-004: borderline
agents = [OptimistAgent(), PessimistAgent(), RegulatorAgent()]
print(f"=== Agent Votes for {app['id']} ===")
for agent in agents:
    v = agent.assess(app)
    flag = ' ⚠️ COMPLIANCE FLAG' if v.compliance_flag else ''
    print(f"  {v.agent_name} (w={v.weight}): {v.vote.value} (conf={v.confidence}){flag}")
    print(f"    Reasoning: {v.reasoning}")

## Section 3: Consensus Mechanisms
Implement 4 mechanisms và compare results.

In [ ]:
def majority_vote(votes: List[AgentVote]) -> Tuple[Decision, float]:
    counts = {d: 0 for d in Decision}
    for v in votes:
        counts[v.vote] += 1
    winner = max(counts, key=counts.get)
    confidence = counts[winner] / len(votes)
    return winner, confidence

def weighted_vote(votes: List[AgentVote]) -> Tuple[Decision, float]:
    weights = {d: 0.0 for d in Decision}
    for v in votes:
        weights[v.vote] += v.weight * v.confidence
    winner = max(weights, key=weights.get)
    total = sum(weights.values())
    confidence = weights[winner] / total if total > 0 else 0
    return winner, round(confidence, 3)

def borda_count(votes: List[AgentVote]) -> Tuple[Decision, float]:
    # Each agent ranks all 3 options based on their vote
    # vote=APPROVE: rank APPROVE 2, REVIEW 1, REJECT 0
    # vote=REVIEW:  rank REVIEW 2, APPROVE 1, REJECT 0
    # vote=REJECT:  rank REJECT 2, REVIEW 1, APPROVE 0
    ranking_map = {
        Decision.APPROVE: {Decision.APPROVE: 2, Decision.REVIEW: 1, Decision.REJECT: 0},
        Decision.REVIEW:  {Decision.APPROVE: 1, Decision.REVIEW: 2, Decision.REJECT: 0},
        Decision.REJECT:  {Decision.APPROVE: 0, Decision.REVIEW: 1, Decision.REJECT: 2},
    }
    borda = {d: 0 for d in Decision}
    for v in votes:
        for d, pts in ranking_map[v.vote].items():
            borda[d] += pts
    winner = max(borda, key=borda.get)
    max_possible = 2 * len(votes)
    confidence = borda[winner] / max_possible
    return winner, round(confidence, 3)

def delphi_simulate(votes: List[AgentVote], app: dict, rounds: int = 2) -> Tuple[Decision, float, int]:
    """Simulate Delphi: agents see each other's votes and can revise."""
    current_votes = list(votes)
    converged_at = 1

    for r in range(1, rounds + 1):
        # Share votes — agents may revise based on compliance flags
        compliance_flags = [v for v in current_votes if v.compliance_flag]
        if compliance_flags:
            # After seeing compliance flag, non-regulator agents tend to revise to REVIEW
            revised = []
            for v in current_votes:
                if not v.compliance_flag and v.vote == Decision.APPROVE:
                    revised.append(AgentVote(v.agent_name, Decision.REVIEW, v.confidence * 0.9,
                        f'Revised after seeing compliance flag from RegulatorAgent', v.weight))
                else:
                    revised.append(v)
            prev_decisions = [v.vote for v in current_votes]
            current_votes = revised
            if [v.vote for v in current_votes] != prev_decisions:
                converged_at = r + 1

    winner, conf = weighted_vote(current_votes)
    return winner, conf, converged_at

# Compare all 4 mechanisms on FC-2024-004
app = APPLICATIONS[3]
votes = [agent.assess(app) for agent in agents]

print(f'=== Consensus Mechanisms Comparison — {app["id"]} ===')
print('\nAgent votes:')
for v in votes:
    print(f'  {v.agent_name}: {v.vote.value} (w={v.weight}, conf={v.confidence})')

print()
m, mc = majority_vote(votes);  print(f'Majority Vote: {m.value} (confidence={mc:.2f})')
w, wc = weighted_vote(votes);  print(f'Weighted Vote: {w.value} (confidence={wc:.3f})')
b, bc = borda_count(votes);    print(f'Borda Count:   {b.value} (confidence={bc:.3f})')
d, dc, dr = delphi_simulate(votes, app); print(f'Delphi:        {d.value} (confidence={dc:.3f}, converged at round {dr})')

## Section 4: Devil's Advocate Pattern
Simulate DA challenge và rebuttal cycle.

In [ ]:
@dataclass
class DebateRound:
    round_num: int
    draft_vote: Decision
    da_challenges: List[str]
    rebuttals: List[str]
    revised_vote: Decision
    changed: bool

# Pre-defined challenges and rebuttals for simulation (no API call needed)
DA_SCENARIOS = {
    'score_borderline': [
        'Score {score} is at lower edge of acceptable range — any minor derogatory event could push below 600',
        'What is the trend of credit score? Declining trend changes risk profile significantly',
    ],
    'dti_stress': [
        'DTI {dti} means 10% income drop pushes DTI above ceiling — no buffer for economic shocks',
        'Existing debt obligations may have hidden balloon payments not captured in DTI calculation',
    ],
    'short_employment': [
        'Employment {emp}mo — has not passed standard 12-month probation period in Vietnamese labor law',
        'Short tenure suggests job instability or career transition — income continuity unproven',
    ]
}

REBUTTALS = {
    'score_borderline': 'Score trend not available in application; recommend request 3-month history as condition',
    'dti_stress': 'Valid concern — recommend income buffer requirement as REVIEW condition',
    'short_employment': 'Internal policy already flags this — collateral requirement addresses the risk',
}

def devil_advocate_round(draft_vote: Decision, app: dict, round_num: int = 1) -> DebateRound:
    score = app['credit_score']
    dti   = app['dti_ratio']
    emp   = app['employment_months']

    challenges = []
    rebuttals = []
    revised_vote = draft_vote

    if score < 680:
        challenges.append(DA_SCENARIOS['score_borderline'][0].format(score=score))
        rebuttals.append(REBUTTALS['score_borderline'])

    if dti > 0.38:
        challenges.append(DA_SCENARIOS['dti_stress'][0].format(dti=dti))
        rebuttals.append(REBUTTALS['dti_stress'])
        # Dti stress concern is valid — if APPROVE, revise to REVIEW
        if draft_vote == Decision.APPROVE:
            revised_vote = Decision.REVIEW

    if emp < 12:
        challenges.append(DA_SCENARIOS['short_employment'][0].format(emp=emp))
        rebuttals.append(REBUTTALS['short_employment'])

    return DebateRound(
        round_num=round_num,
        draft_vote=draft_vote,
        da_challenges=challenges,
        rebuttals=rebuttals,
        revised_vote=revised_vote,
        changed=(revised_vote != draft_vote)
    )

# Simulate DA round for FC-2024-004
app = APPLICATIONS[3]
draft, _ = weighted_vote(votes)
print(f'=== Devil\'s Advocate Round — {app["id"]} ===')
print(f'Draft decision: {draft.value}')

da_round = devil_advocate_round(draft, app, round_num=1)
print(f'\nDA Challenges ({len(da_round.da_challenges)}):')
for c in da_round.da_challenges:
    print(f'  ⚔️  {c}')
print(f'\nRebuttals ({len(da_round.rebuttals)}):')
for r in da_round.rebuttals:
    print(f'  🛡️  {r}')
print(f'\nRevised decision: {da_round.revised_vote.value} (changed: {da_round.changed})')

## Section 5: Meta-Arbitrator với Audit Trail
Tổng hợp tất cả thành final decision có đầy đủ audit documentation.

In [ ]:
@dataclass
class PanelDecision:
    application_id: str
    final_decision: Decision
    consensus_method: str
    panel_confidence: float
    explanation: str
    conditions: List[str]
    dissent_summary: str
    audit_trail: List[dict]  # Per-agent data for regulatory audit
    da_challenges_addressed: List[str] = field(default_factory=list)

def meta_arbitrate(app: dict, votes: List[AgentVote], da_round: Optional[DebateRound] = None) -> PanelDecision:
    app_id = app['id']

    # Check regulatory veto FIRST
    compliance_flags = [v for v in votes if v.compliance_flag]
    if compliance_flags:
        cf = compliance_flags[0]
        return PanelDecision(
            application_id=app_id,
            final_decision=Decision.REVIEW,
            consensus_method='regulatory_veto',
            panel_confidence=0.99,
            explanation=f'Regulatory veto by {cf.agent_name}: {cf.reasoning}',
            conditions=['Resolve compliance issue before proceeding', 'Re-submit with complete documentation'],
            dissent_summary='Regulatory veto overrides all other votes',
            audit_trail=[{'agent': v.agent_name, 'vote': v.vote.value, 'weight': v.weight,
                         'confidence': v.confidence, 'reasoning': v.reasoning[:120]} for v in votes]
        )

    # Apply DA revision if available
    effective_votes = votes
    da_addressed = []
    if da_round and da_round.changed:
        # Revise votes that changed due to DA
        effective_votes = []
        for v in votes:
            if v.agent_name == 'OptimistAgent' and da_round.draft_vote == v.vote:
                effective_votes.append(AgentVote(v.agent_name, da_round.revised_vote,
                    v.confidence * 0.9, f'Revised after DA: {da_round.da_challenges[0][:60]}', v.weight))
            else:
                effective_votes.append(v)
        da_addressed = da_round.da_challenges[:2]

    # Weighted vote on effective votes
    winner, conf = weighted_vote(effective_votes)

    # Build conditions
    conditions = []
    if winner == Decision.REVIEW:
        if app['employment_months'] < 12:
            conditions.append('Require 1 collateral asset (employment < 12 months)')
        if app['dti_ratio'] > 0.40:
            conditions.append('Provide 6-month bank statements to verify income stability')
        conditions.append('Re-verify loan purpose with supporting documents')

    # Dissent summary
    majority_decision = winner
    dissenters = [v for v in effective_votes if v.vote != majority_decision]
    dissent = ', '.join([f'{d.agent_name}: {d.vote.value}' for d in dissenters]) if dissenters else 'None — unanimous'

    # Build explanation
    vote_summary = ', '.join([f'{v.agent_name}={v.vote.value}(w{v.weight})' for v in effective_votes])
    explanation = f'Weighted panel ({vote_summary}) → {winner.value} (confidence={conf:.2f})'
    if da_addressed:
        explanation += f'. DA challenges addressed: {len(da_addressed)} concern(s) incorporated.'

    return PanelDecision(
        application_id=app_id,
        final_decision=winner,
        consensus_method='weighted_vote_with_da',
        panel_confidence=conf,
        explanation=explanation,
        conditions=conditions,
        dissent_summary=dissent,
        audit_trail=[{'agent': v.agent_name, 'vote': v.vote.value, 'weight': v.weight,
                     'confidence': v.confidence} for v in effective_votes],
        da_challenges_addressed=da_addressed
    )

# Full panel for FC-2024-004
app  = APPLICATIONS[3]
votes = [agent.assess(app) for agent in agents]
da   = devil_advocate_round(Decision.APPROVE, app)
decision = meta_arbitrate(app, votes, da)

print(f'=== Panel Decision: {decision.application_id} ===')
print(f'Final: {decision.final_decision.value}')
print(f'Method: {decision.consensus_method}')
print(f'Confidence: {decision.panel_confidence:.3f}')
print(f'Explanation: {decision.explanation}')
print(f'Conditions: {decision.conditions}')
print(f'Dissent: {decision.dissent_summary}')
print(f'Audit trail entries: {len(decision.audit_trail)}')

## Section 6: Panel Quality Metrics
Đo diversity index và per-agent accuracy simulation.

In [ ]:
def diversity_index(votes: List[AgentVote]) -> float:
    """Shannon entropy of vote distribution — 0 = all agree, max = all different."""
    counts = {d: 0 for d in Decision}
    for v in votes:
        counts[v.vote] += 1
    n = len(votes)
    entropy = 0.0
    for c in counts.values():
        if c > 0:
            p = c / n
            entropy -= p * math.log2(p)
    return round(entropy, 3)

def simulate_panel_accuracy(n_cases: int = 100) -> dict:
    """Simulate panel vs single-agent accuracy on random borderline cases."""
    random.seed(42)
    panel_correct = 0
    single_correct = 0

    agents_sim = [OptimistAgent(), PessimistAgent(), RegulatorAgent()]

    for _ in range(n_cases):
        # Generate random borderline case
        score = random.randint(570, 700)
        dti   = round(random.uniform(0.35, 0.48), 2)
        emp   = random.randint(6, 24)
        app = {'id': 'SIM', 'credit_score': score, 'dti_ratio': dti,
               'employment_months': emp, 'loan_amount': 400_000_000, 'loan_purpose': 'business vehicle'}

        # Ground truth: complex heuristic
        if score >= 650 and dti <= 0.42 and emp >= 12:
            ground_truth = Decision.APPROVE
        elif score < 600 or dti > 0.45:
            ground_truth = Decision.REJECT
        else:
            ground_truth = Decision.REVIEW

        # Panel decision
        panel_votes = [a.assess(app) for a in agents_sim]
        panel_decision, _ = weighted_vote(panel_votes)

        # Single agent (Balanced — middle ground)
        if score >= 650 and dti <= 0.42:
            single_decision = Decision.APPROVE
        elif score < 600:
            single_decision = Decision.REJECT
        else:
            # Single agent is less nuanced — often misses REVIEW
            single_decision = Decision.APPROVE if random.random() > 0.55 else Decision.REJECT

        if panel_decision == ground_truth:  panel_correct += 1
        if single_decision == ground_truth: single_correct += 1

    return {
        'panel_accuracy': round(panel_correct / n_cases, 3),
        'single_accuracy': round(single_correct / n_cases, 3),
        'improvement': round((panel_correct - single_correct) / n_cases, 3)
    }

# Run metrics
print('=== Panel Quality Metrics ===')
for i, app in enumerate(APPLICATIONS):
    v = [a.assess(app) for a in agents]
    route = route_to_debate(app)
    if route.use_debate:
        di = diversity_index(v)
        print(f"{app['id']}: diversity_index={di:.3f} ({'good' if di > 0.5 else 'low — echo chamber risk'})")
        for vote in v:
            print(f'  {vote.agent_name}: {vote.vote.value}')

print('\n=== Accuracy Simulation (100 borderline cases) ===')
results = simulate_panel_accuracy(100)
print(f'Panel accuracy:  {results["panel_accuracy"]:.1%}')
print(f'Single accuracy: {results["single_accuracy"]:.1%}')
print(f'Improvement:     +{results["improvement"]:.1%} percentage points')

## Section 7: LoanBotDebateSystem — TODO và Solution

Implement `LoanBotDebateSystem` kết hợp tất cả components từ Sections 1-6.

In [ ]:
# TODO: Implement LoanBotDebateSystem
class LoanBotDebateSystemTODO:
    """
    Complete multi-agent debate system for FinTech Corp.

    Components to integrate:
    1. route_to_debate()       — Section 1
    2. Agent personas          — Section 2
    3. weighted_vote()         — Section 3
    4. devil_advocate_round()  — Section 4
    5. meta_arbitrate()        — Section 5
    6. diversity_index()       — Section 6

    TODO: Implement decide() method that:
    a. Route application — if no debate needed, return direct_decision
    b. Run agent panel (3 or 5 agents based on route)
    c. Check if agents are diverse enough (diversity_index > 0.3) — warn if low
    d. Run devil's advocate round on initial weighted vote
    e. Meta-arbitrate final decision
    f. Return PanelDecision with full audit trail
    """

    def __init__(self):
        # TODO: initialize agent instances
        pass

    def decide(self, app: dict) -> PanelDecision:
        # TODO: implement using components above
        raise NotImplementedError('Implement me!')

print('TODO class defined. Implement decide() method.')

In [ ]:
# SOLUTION: LoanBotDebateSystem
class LoanBotDebateSystem:
    """Complete multi-agent debate system for FinTech Corp."""

    def __init__(self):
        self.agents = [OptimistAgent(), PessimistAgent(), RegulatorAgent()]
        self.decisions_log = []  # For tracking quality metrics

    def decide(self, app: dict) -> PanelDecision:
        app_id = app['id']

        # Step 1: Route — debate needed?
        route = route_to_debate(app)
        if not route.use_debate:
            direct = PanelDecision(
                application_id=app_id,
                final_decision=route.direct_decision,
                consensus_method='direct_rule',
                panel_confidence=0.97,
                explanation=f'Direct decision: {route.reason}',
                conditions=[],
                dissent_summary='No debate — clear criteria met',
                audit_trail=[{'route_reason': route.reason}]
            )
            self.decisions_log.append({'app_id': app_id, 'decision': direct})
            return direct

        # Step 2: Agent panel votes
        active_agents = self.agents[:route.n_agents]
        votes = [agent.assess(app) for agent in active_agents]

        # Step 3: Check diversity
        di = diversity_index(votes)
        if di < 0.3:
            print(f'  ⚠️  Low diversity ({di}) for {app_id} — panel may be echo chamber')

        # Step 4: Devil's Advocate on initial weighted vote
        draft_decision, _ = weighted_vote(votes)
        da = devil_advocate_round(draft_decision, app)

        # Step 5: Meta-Arbitrate
        decision = meta_arbitrate(app, votes, da)

        # Log for quality tracking
        self.decisions_log.append({
            'app_id': app_id,
            'diversity_index': di,
            'n_agents': route.n_agents,
            'decision': decision
        })

        return decision

    def quality_report(self):
        print(f'=== Panel Quality Report ===')
        print(f'Total decisions: {len(self.decisions_log)}')
        debate_cases = [d for d in self.decisions_log if 'diversity_index' in d]
        if debate_cases:
            avg_di = sum(d['diversity_index'] for d in debate_cases) / len(debate_cases)
            print(f'Debate cases: {len(debate_cases)}')
            print(f'Average diversity index: {avg_di:.3f}')

# Run full system on all 4 applications
system = LoanBotDebateSystem()
print('=== LoanBot Debate System — All Applications ===')
print()
for app in APPLICATIONS:
    decision = system.decide(app)
    print(f'{decision.application_id}: {decision.final_decision.value} ({decision.consensus_method}, conf={decision.panel_confidence:.3f})')
    if decision.conditions:
        for c in decision.conditions:
            print(f'  Condition: {c}')
    print(f'  Explanation: {decision.explanation[:100]}...')
    print()

print()
system.quality_report()